In [115]:
import pandas as pd

from MATRIX.utils import get_data, get_metrics, get_results
from MATRIX.models import BaseSurvival, CoxRegression, DeepSurv, RandomSurvForest, DeepMultiTask, CoxRegressionWithTimeVarying, DeepTimeVarying

# Experiments

## Survival analysis (standard)

### Get data

In [116]:
X_train, y_train, X_validation, y_validation, X_test, y_test, feature_names, scaler = get_data(dataset_name="colon.csv")


- - - - colon.csv (csv) - - - -

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 911 entries, 0 to 910
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   event         911 non-null    int64  
 1   time          911 non-null    int64  
 2   num_age       911 non-null    int64  
 3   num_nodes     911 non-null    float64
 4   fac_rx        911 non-null    object 
 5   fac_sex       911 non-null    int64  
 6   fac_differ    911 non-null    object 
 7   fac_obstruct  911 non-null    int64  
 8   fac_perfor    911 non-null    int64  
 9   fac_adhere    911 non-null    int64  
 10  fac_node4     911 non-null    int64  
dtypes: float64(1), int64(8), object(2)
memory usage: 78.4+ KB

             event         time     num_age   num_nodes fac_rx     fac_sex  \
count   911.000000   911.000000  911.000000  911.000000    911  911.000000   
unique         NaN          NaN         NaN         NaN      3         NaN   
top 

### Instantiate models

In [117]:
modelCoxRegression = CoxRegression()
modelRandomSurvForest = RandomSurvForest(0)
modelDeepSurv = DeepSurv(X_train.shape[1])

### Fit and predict

In [118]:
modelCoxRegression = modelCoxRegression.fit(X_train, y_train)
modelRandomSurvForest = modelRandomSurvForest.fit(X_train, y_train)
modelDeepSurv = modelDeepSurv.fit(X_train, y_train)

survPredCoxRegression = modelCoxRegression.predict(X_test)
survPredRandomSurvForest = modelRandomSurvForest.predict(X_test)
survPredDeepSurv = modelDeepSurv.predict(X_test)

### Metrics

In [119]:
get_metrics([y_train, y_test], [survPredCoxRegression])

{'C-Index Harrel': 0.620846552012022,
 'C-Index IPCW': 0.6122561689357433,
 'Cumulative Dinamic AUC': [0.6285741381350445,
  0.6553820831596768,
  0.6558112662759072]}

In [120]:
get_metrics([y_train, y_test], [survPredRandomSurvForest])

{'C-Index Harrel': 0.6361245616964435,
 'C-Index IPCW': 0.6280965938985875,
 'Cumulative Dinamic AUC': [0.6317437061063123,
  0.6761228984690368,
  0.6878876204225357]}

In [121]:
get_metrics([y_train, y_test], [survPredDeepSurv])

{'C-Index Harrel': 0.47591417598931374,
 'C-Index IPCW': 0.47620015393083887,
 'Cumulative Dinamic AUC': [0.4634080062550316,
  0.4657317803643588,
  0.4733613938044623]}

### Survival / Cumulative Hazard Functions

In [122]:
survival_function = modelCoxRegression.predict_survival_function(X=X_train, estimator_name="modelCoxRegression", dataset="colon.csv", seed=0, plot=True)

In [123]:
cumulative_hazard_function = modelCoxRegression.predict_cumulative_hazard_function(X=X_train, estimator_name="modelCoxRegression", dataset="colon.csv", seed=0, plot=True)

### eXplainable Artificial Intelligence

In [124]:
modelCoxRegression.calculate_xai(X=X_train, estimator_name="modelCoxRegression", dataset="colon.csv", seed=0, feature_names=feature_names, background=50, plot=True)

## Survival analysis (multitask)

### Get data

In [125]:
X_train, y_train, X_validation, y_validation, X_test, y_test, feature_names, scaler = get_data(dataset_name="colon.csv", to_multitask=True)


- - - - colon.csv (csv) - - - -

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 911 entries, 0 to 910
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   event         911 non-null    int64  
 1   time          911 non-null    int64  
 2   num_age       911 non-null    int64  
 3   num_nodes     911 non-null    float64
 4   fac_rx        911 non-null    object 
 5   fac_sex       911 non-null    int64  
 6   fac_differ    911 non-null    object 
 7   fac_obstruct  911 non-null    int64  
 8   fac_perfor    911 non-null    int64  
 9   fac_adhere    911 non-null    int64  
 10  fac_node4     911 non-null    int64  
dtypes: float64(1), int64(8), object(2)
memory usage: 78.4+ KB

             event         time     num_age   num_nodes fac_rx     fac_sex  \
count   911.000000   911.000000  911.000000  911.000000    911  911.000000   
unique         NaN          NaN         NaN         NaN      3         NaN   
top 

### Instantiate models

In [126]:
modelDeepMultiTask = DeepMultiTask(X_train.shape[1])

### Fit and predict

In [127]:
modelDeepMultiTask = modelDeepMultiTask.fit(X_train, y_train)
survPredDeepMultiTask, binPredDeepMultiTask = modelDeepMultiTask.predict(X_test)

### Metrics

In [128]:
get_metrics([y_train, y_test], [survPredDeepMultiTask, binPredDeepMultiTask])

{'C-Index Harrel': 0.4756637168141593,
 'C-Index IPCW': 0.4600091249848916,
 'Cumulative Dinamic AUC': [0.507637820668982,
  0.4911281585982256,
  0.41985537745060636],
 'MAE': 0.5519125683060109,
 'AMAE': 0.5519125683060109,
 'MS': 0.14754098360655737,
 'CCR': 0.44808743169398907,
 'RECALL0': 0.14754098360655737,
 'RECALL1': 0.3005464480874317}

### Survival / Cumulative Hazard Functions

In [129]:
survival_function = modelDeepMultiTask.predict_survival_function(X=X_train, estimator_name="modelDeepMultiTask", dataset="colon.csv", seed=0, plot=False)

In [130]:
cumulative_hazard_function = modelDeepMultiTask.predict_cumulative_hazard_function(X=X_train, estimator_name="modelDeepMultiTask", dataset="colon.csv", seed=0, plot=False)

### eXplainable Artificial Intelligence

In [131]:
modelDeepMultiTask.calculate_xai(X=X_train, estimator_name="modelDeepMultiTask", dataset="colon.csv", seed=0, feature_names=feature_names, background=50, plot=False)

## Survival analysis (time varying)

In [132]:
df = pd.read_csv("./MATRIX/datasets/colon.csv")

### Find splits dynamically

In [133]:
splits = BaseSurvival.dinamic_discretise(y=df[["time", "event"]], dataset="colon", seed=0, plot=False)

In [134]:
df["identifier"] = df.index.values

### Transform to time varying format

In [135]:
df = BaseSurvival.to_time_dependent(dataframe=df, splits=splits, identifier="identifier", time="time", event="event")

In [136]:
df = BaseSurvival.to_time_varying(dataframe=df, identifier="identifier", time="time", event="event")

### Get data

In [137]:
X_train, y_train, X_validation, y_validation, X_test, y_test, feature_names, scaler = get_data(df=df)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4049 entries, 0 to 4048
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   identifier    4049 non-null   int64  
 1   num_age       4049 non-null   int64  
 2   num_nodes     4049 non-null   float64
 3   fac_rx        4049 non-null   object 
 4   fac_sex       4049 non-null   int64  
 5   fac_differ    4049 non-null   object 
 6   fac_obstruct  4049 non-null   int64  
 7   fac_perfor    4049 non-null   int64  
 8   fac_adhere    4049 non-null   int64  
 9   fac_node4     4049 non-null   int64  
 10  time_frame    4049 non-null   int64  
 11  days_risk     4049 non-null   float64
 12  event         4049 non-null   int64  
 13  time_start    4049 non-null   float64
 14  time_stop     4049 non-null   float64
dtypes: float64(4), int64(9), object(2)
memory usage: 474.6+ KB

         identifier      num_age    num_nodes   fac_rx      fac_sex  \
count   4049.000000  4049.

### Instantiate models

In [138]:
modelCoxRegressionWithTimeVarying = CoxRegressionWithTimeVarying()
modelDeepTimeVarying = DeepTimeVarying(X_train.shape[1])

### Fit and predict

In [139]:
modelCoxRegressionWithTimeVarying = modelCoxRegressionWithTimeVarying.fit(X_train, y_train)
modelDeepTimeVarying = modelDeepTimeVarying.fit(X_train, y_train)

survPredCoxRegressionWithTimeVarying = modelCoxRegressionWithTimeVarying.predict(X_test)
survPredDeepTimeVarying = modelDeepTimeVarying.predict(X_test)

### Metrics

In [140]:
get_metrics([y_train, y_test], [survPredCoxRegressionWithTimeVarying])

{'C-Index Harrel': 0.5240485829959514,
 'C-Index IPCW': 0.5662865225788647,
 'Cumulative Dinamic AUC': [0.4258347409507907,
  0.4722733737449364,
  0.48463033105518194]}

In [141]:
get_metrics([y_train, y_test], [survPredDeepTimeVarying])

{'C-Index Harrel': 0.5978461538461538,
 'C-Index IPCW': 0.6072924061525677,
 'Cumulative Dinamic AUC': [0.5358466625318828,
  0.5839364482979337,
  0.5671598308273877]}

### Survival / Cumulative Hazard Functions

In [142]:
survival_function = modelCoxRegressionWithTimeVarying.predict_survival_function(X=X_train, estimator_name="modelCoxRegressionWithTimeVarying", dataset="colon.csv", seed=0, plot=False)

In [143]:
cumulative_hazard_function = modelCoxRegressionWithTimeVarying.predict_cumulative_hazard_function(X=X_train, estimator_name="modelCoxRegressionWithTimeVarying", dataset="colon.csv", seed=0, plot=False)

### eXplainable Artificial Intelligence

In [144]:
modelCoxRegressionWithTimeVarying.calculate_xai(X=X_train, estimator_name="modelCoxRegressionWithTimeVarying", dataset="colon.csv", seed=0, feature_names=feature_names, background=50, plot=False)

# Results

In [150]:
#results = get_results(estimator_name="CoxRegression", dataset="colon.csv")
results = get_results(estimator_name="CoxRegressionWithTimeVarying", dataset="cgd.csv")
#results = get_results(estimator_name="DeepMultiTaskFFNN", dataset="colon.csv")

In [146]:
for i, result in enumerate(results):
    print(f"Result {i}:\n")
    print(f"    * Estimator: {result.config['estimator_name']} - Dataset: {result.config['dataset']} - Seed: {result.config['random_state']}")
    print(f"    * Best Params: {result.data_.best_params}")
    print()
    print(f"    * Metrics: {get_metrics(result.data_.targets, result.data_.predictions)}")
    print("\n\n")

Result 0:

    * Estimator: CoxRegressionWithTimeVarying - Dataset: cgd.csv - Seed: 0
    * Best Params: {'penalizer': 10.0, 'l1_ratio': 0.0}

    * Metrics: {'C-Index Harrel': 0.4361979166666667, 'C-Index IPCW': 0.5732415376912047, 'Cumulative Dinamic AUC': [0.39160681325664665, 0.26038124872553725, 0.36137667027024956]}



Result 1:

    * Estimator: CoxRegressionWithTimeVarying - Dataset: cgd.csv - Seed: 1
    * Best Params: {'penalizer': 0.1, 'l1_ratio': 1.0}

    * Metrics: {'C-Index Harrel': 0.4247881355932203, 'C-Index IPCW': 0.4783095881950735, 'Cumulative Dinamic AUC': [0.5, 0.5, 0.5]}



Result 2:

    * Estimator: CoxRegressionWithTimeVarying - Dataset: cgd.csv - Seed: 2
    * Best Params: {'penalizer': 1.0, 'l1_ratio': 0.0}

    * Metrics: {'C-Index Harrel': 0.625886524822695, 'C-Index IPCW': 0.7409385758205188, 'Cumulative Dinamic AUC': [0.2650043442474134, 0.476005688192635, 0.6126884567282346]}



Result 3:

    * Estimator: CoxRegressionWithTimeVarying - Dataset: cgd.cs